# Plan Step 4 v2: NN eval 学習 改善版 (Colab T4 GPU)

**v1 (= colab_train_nn_eval.ipynb) との差分**:
- epoch 20 → **100** (= early stopping 付き)
- lr 1e-3 固定 → **cosine annealing** (= 1e-3 → 1e-5 で滑らかに減少)
- batch 512 → **1024** (= T4 メモリ範囲、 step 数減らして loss 振動緩和)
- early stopping: best val_sign_acc が **15 epoch 改善しなければ停止**
- best モデル + 学習曲線も Drive に保存 (= 後解析用)

**期待**: v1 best val_sign_acc 0.643 → 0.68-0.72。

## 使い方 (= v1 と同じ)

1. Drive `MyDrive/onepiece_nn/self_play_snapshots.jsonl` を準備 (= v1 で既にアップ済)
2. Colab で開く → 「ランタイム > ランタイムのタイプを変更 > T4 GPU」
3. 「ランタイム > すべてのセルを実行」
4. 完了後 Drive の `nn_eval_v2.pt` + `_nn_feature_keys_v2.json` + `training_curve_v2.json` をローカル DL

T4 で約 15-30 分の見込み。

## 1. GPU 確認

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)
assert torch.cuda.is_available(), 'GPU が有効化されていません。 ランタイム > ランタイムのタイプを変更 > T4 GPU を選択してください。'

## 2. Google Drive マウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/onepiece_nn'
SNAPSHOT_PATH = os.path.join(WORK_DIR, 'self_play_snapshots.jsonl')
OUTPUT_PT = os.path.join(WORK_DIR, 'nn_eval_v2.pt')
OUTPUT_KEYS = os.path.join(WORK_DIR, '_nn_feature_keys_v2.json')
OUTPUT_CURVE = os.path.join(WORK_DIR, 'training_curve_v2.json')

assert os.path.exists(SNAPSHOT_PATH), f'snapshot が見つからない: {SNAPSHOT_PATH}'
print('snapshot size:', os.path.getsize(SNAPSHOT_PATH) // (1024*1024), 'MB')

## 3. snapshot ロード + tensor 化

In [ ]:
import json, time
import numpy as np

N_ACTION_CATEGORIES = 9

t0 = time.time()
snapshots = []
feature_keys = None
with open(SNAPSHOT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            snap = json.loads(line)
        except json.JSONDecodeError:
            continue
        snapshots.append(snap)
        if feature_keys is None and 'features' in snap:
            feature_keys = sorted(snap['features'].keys())
print(f'loaded {len(snapshots):,} snapshots in {time.time()-t0:.1f}s, dim={len(feature_keys)}')

device = torch.device('cuda')
X_np = np.zeros((len(snapshots), len(feature_keys)), dtype=np.float32)
y_np = np.zeros((len(snapshots),), dtype=np.float32)
for i, snap in enumerate(snapshots):
    feats = snap.get('features', {})
    for j, k in enumerate(feature_keys):
        X_np[i, j] = float(feats.get(k, 0.0))
    y_np[i] = float(snap.get('winner', snap.get('final_winner', 0)))
X = torch.from_numpy(X_np).to(device)
y = torch.from_numpy(y_np).unsqueeze(1).to(device)
a = torch.zeros(len(snapshots), dtype=torch.long, device=device)

n = len(snapshots)
split = int(n * 0.8)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
a_train, a_val = a[:split], a[split:]
print(f'train: {X_train.shape[0]:,}, val: {X_val.shape[0]:,}')
del snapshots, X_np, y_np  # メモリ解放

## 4. モデル定義 (= v1 と同じ SimpleEvaluator)

In [ ]:
import torch.nn as nn

class SimpleEvaluator(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.value_head = nn.Linear(hidden_dim // 2, 1)
        self.policy_head = nn.Linear(hidden_dim // 2, N_ACTION_CATEGORIES)
    def forward(self, x):
        h = self.shared(x)
        return self.value_head(h), self.policy_head(h)

model = SimpleEvaluator(input_dim=len(feature_keys)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'params: {n_params:,}')

## 5. 学習 (= cosine scheduler + early stopping)

In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

EPOCHS = 100
BATCH = 1024
LR_MAX = 1e-3
LR_MIN = 1e-5
EARLY_STOP_PATIENCE = 15

train_ds = TensorDataset(X_train, y_train, a_train)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)

optimizer = optim.Adam(model.parameters(), lr=LR_MAX)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
value_criterion = nn.MSELoss()
policy_criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
epochs_since_best = 0
curve = []  # [{epoch, train_loss, val_sign_acc, lr}]

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0
    n_batches = 0
    for Xb, yb, ab in train_loader:
        optimizer.zero_grad()
        v_pred, p_pred = model(Xb)
        loss_v = value_criterion(v_pred, yb)
        loss_p = policy_criterion(p_pred, ab)
        loss = loss_v + 0.5 * loss_p
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item()
        n_batches += 1

    model.eval()
    with torch.no_grad():
        v_pred_val, _ = model(X_val)
        sign_acc = float((torch.sign(v_pred_val) == torch.sign(y_val)).float().mean().item())
    avg_loss = train_loss_sum / max(1, n_batches)
    current_lr = optimizer.param_groups[0]['lr']
    curve.append({'epoch': epoch+1, 'train_loss': avg_loss, 'val_sign_acc': sign_acc, 'lr': current_lr})
    msg = f'epoch {epoch+1:3d}/{EPOCHS}: train_loss={avg_loss:.4f}, val_sign_acc={sign_acc:.3f}, lr={current_lr:.2e}'

    if sign_acc > best_val_acc:
        best_val_acc = sign_acc
        torch.save({k: v.cpu() for k, v in model.state_dict().items()}, OUTPUT_PT)
        with open(OUTPUT_KEYS, 'w', encoding='utf-8') as f:
            json.dump(feature_keys, f, ensure_ascii=False)
        epochs_since_best = 0
        msg += f'  ✔ saved (best={sign_acc:.3f})'
    else:
        epochs_since_best += 1
    print(msg)

    scheduler.step()

    if epochs_since_best >= EARLY_STOP_PATIENCE:
        print(f'\n=== early stopping at epoch {epoch+1} (no improvement for {EARLY_STOP_PATIENCE} epochs) ===')
        break

with open(OUTPUT_CURVE, 'w', encoding='utf-8') as f:
    json.dump(curve, f, ensure_ascii=False, indent=2)
print(f'\n=== DONE. best val_sign_acc={best_val_acc:.3f} ===')

## 6. 成果物確認 + 学習曲線可視化

In [ ]:
import matplotlib.pyplot as plt

for p in [OUTPUT_PT, OUTPUT_KEYS, OUTPUT_CURVE]:
    if os.path.exists(p):
        print(f'  {p}: {os.path.getsize(p):,} bytes')
    else:
        print(f'  {p}: NOT FOUND')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_x = [c['epoch'] for c in curve]
ax1.plot(epochs_x, [c['train_loss'] for c in curve], label='train_loss')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend(); ax1.set_yscale('log')
ax2.plot(epochs_x, [c['val_sign_acc'] for c in curve], label='val_sign_acc', color='orange')
ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='coin flip')
ax2.axhline(best_val_acc, color='green', linestyle='--', alpha=0.5, label=f'best={best_val_acc:.3f}')
ax2.set_xlabel('epoch'); ax2.set_ylabel('val sign acc'); ax2.legend()
plt.tight_layout(); plt.show()

print('\n--- 次の手順 ---')
print('1. Drive の nn_eval_v2.pt と _nn_feature_keys_v2.json をローカルにダウンロード')
print('2. ローカルで以下のように rename して既存と差し替え:')
print('   mv ~/Downloads/nn_eval_v2.pt ~/projects/onepiece_research/db/nn_eval.pt')
print('   mv ~/Downloads/_nn_feature_keys_v2.json ~/projects/onepiece_research/db/_nn_feature_keys.json')